# 04 · Modelo, avaliação e predição

Aqui o enunciado e literal: *"criar, treinar e avaliar um modelo usando **puramente
BigQuery ML (SQL)**"*. Nao existe versao em Python desta etapa.

---

### O TRANSFORM e o centro do projeto

As nove features nascem dentro do `CREATE MODEL`. O `TRANSFORM` e serializado
junto com o modelo, entao quando ele for para o Vertex AI em outubro a API vai
aplicar **exatamente este codigo** — nao uma reimplementacao que diverge com o
tempo. E a decisao que evita o maior retrabalho do Trabalho 2.

Leia a celula do v1 com atencao: e o pedaco de SQL mais importante do trabalho.

In [ ]:
import pandas as pd
from google.cloud import bigquery

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery db-dtypes

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## Parâmetros

In [ ]:
PROJECT_ID = "fraudflow-pdm-gps"
client = bigquery.Client(project=PROJECT_ID)
print(PROJECT_ID)

## v1 · regressão logística

O baseline. Sem ele, o numero do v2 nao significa nada. Leva alguns minutos.

In [ ]:
SQL_V1 = f"""
-- ===========================================================================
-- 30 · Modelo v1 — regressão logística (baseline)
--
-- TODAS as features do modelo nascem aqui dentro, no TRANSFORM. Esse é o
-- ponto central da arquitetura: o TRANSFORM é serializado junto com o modelo,
-- então quando ele for para o Vertex AI em outubro, a API vai aplicar
-- exatamente este código — não uma reimplementação em Python que diverge.
--
-- A entrada é gold.ml_input, que é o espelho do contrato do evento do T2.
-- ===========================================================================

CREATE OR REPLACE MODEL `{PROJECT_ID}.gold.fraud_logistic`

TRANSFORM (
  ------------------------------------------------------------------ features
  amount,
  LN(amount)                                              AS log_amount,
  category,

  EXTRACT(HOUR      FROM transaction_ts)                  AS hour,
  EXTRACT(DAYOFWEEK FROM transaction_ts)                  AS day_of_week,
  (EXTRACT(HOUR FROM transaction_ts) >= 22
   OR EXTRACT(HOUR FROM transaction_ts) <= 3)             AS is_night,

  -- Idade completa na data da compra. O DATE_DIFF com YEAR conta viradas de
  -- ano, não anos completos, então descontamos 1 quando o aniversário ainda
  -- não passou. Calculada a partir de transaction_ts — NUNCA do unix_time,
  -- que deixaria todo mundo sete anos mais novo sem emitir erro.
  DATE_DIFF(DATE(transaction_ts), customer_dob, YEAR)
    - IF(FORMAT_DATE('%m%d', DATE(transaction_ts))
          < FORMAT_DATE('%m%d', customer_dob), 1, 0)      AS age,

  -- Haversine em km. Escrito com ATAN2/SIN/COS/SQRT/POW de proposito:
  -- ST_DISTANCE e ST_GEOGPOINT NAO constam na lista de funcoes que o BigQuery
  -- aceita dentro do TRANSFORM ao exportar/implantar o modelo, e GEOGRAPHY e
  -- um tipo proibido nesse caminho. Com ST_* o modelo TREINA normalmente, mas
  -- o deploy no Vertex AI em outubro fica em risco. ACOS(-1) e o pi.
  2 * 6371.0 * ATAN2(
    SQRT(
      POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
      + COS(customer_lat * ACOS(-1) / 180)
        * COS(merchant_lat * ACOS(-1) / 180)
        * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2)
    ),
    SQRT(1 -
      ( POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
        + COS(customer_lat * ACOS(-1) / 180)
          * COS(merchant_lat * ACOS(-1) / 180)
          * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2) )
    )
  )                                                       AS distance_km,

  city_pop,

  --------------------------------------------------- passagem sem transformar
  -- Coluna de corte temporal. Precisa sair do TRANSFORM sem alteração para
  -- o DATA_SPLIT_COL enxergá-la. Não é feature.
  transaction_ts,

  -- Rótulo.
  is_fraud
)

OPTIONS (
  MODEL_TYPE               = 'LOGISTIC_REG',
  INPUT_LABEL_COLS         = ['is_fraud'],

  -- Fraude é ~0,5% dos casos. Sem reponderar, o modelo aprende a responder
  -- "não é fraude" sempre e exibe 99,5% de acurácia sem ter aprendido nada.
  AUTO_CLASS_WEIGHTS       = TRUE,

  -- Recorte temporal: ordena pela coluna indicada e reserva as últimas linhas
  -- para validação. Treina no passado, valida no período mais recente.
  DATA_SPLIT_METHOD        = 'SEQ',
  DATA_SPLIT_COL           = 'transaction_ts',
  DATA_SPLIT_EVAL_FRACTION = 0.20,

  -- Já registra e versiona no Vertex AI. Não custa nada enquanto não houver
  -- endpoint implantado, e adianta metade do Trabalho 2.
  MODEL_REGISTRY           = 'VERTEX_AI',
  VERTEX_AI_MODEL_ID       = 'fraudflow-logistic'
) AS
SELECT * EXCEPT (transaction_id, split_key)
FROM `{PROJECT_ID}.gold.ml_input`;
"""

In [ ]:
client.query(SQL_V1).result()
print("modelo v1 treinado")

### Se o registro no Vertex AI falhar

Ha uma incerteza tecnica conhecida: `TIMESTAMP` na posicao de `DATA_SPLIT_COL`
dentro do `TRANSFORM` pode ser recusado na exportacao para o Vertex AI.

A `gold.ml_input` ja traz a coluna `split_key` pronta — o mesmo instante em texto,
no formato `'YYYY-MM-DD HH:MM:SS'`, cuja ordem alfabetica e a ordem cronologica.
O recorte temporal fica identico e o TIMESTAMP sai da entrada do modelo.

Para trocar, no `SQL_V1` acima:

1. no `TRANSFORM`, troque a linha `transaction_ts,` por
   `FORMAT_TIMESTAMP('%Y-%m-%d %H:%M:%S', transaction_ts) AS split_key,`
2. em `OPTIONS`, troque `DATA_SPLIT_COL = 'transaction_ts'` por `'split_key'`
3. no `SELECT` final, mantenha `EXCEPT (transaction_id, split_key)` como esta

## v2 · árvores impulsionadas

Mesmo `TRANSFORM`, de proposito: a comparacao so e honesta se as features forem
identicas. O que muda e o algoritmo.

**Atencao ao custo.** Treino de modelo tem tarifa propria, nao entra na cota
gratuita, e o boosted tree tem um componente cobrado a parte. E o unico gasto
imprevisivel do Trabalho 1 — abra o Billing logo depois.

In [ ]:
SQL_V2 = f"""
-- ===========================================================================
-- 31 · Modelo v2 — árvores impulsionadas
--
-- TRANSFORM idêntico ao do v1, de propósito: a comparação entre os dois
-- modelos só é honesta se as features forem exatamente as mesmas. O que muda
-- é o algoritmo.
--
-- CUSTO: este é o único gasto imprevisível do Trabalho 1. Treino de modelo
-- tem tarifa própria e não entra na cota gratuita, e o boosted tree tem um
-- componente cobrado à parte. Abra o Billing logo depois deste treino.
-- ===========================================================================

CREATE OR REPLACE MODEL `{PROJECT_ID}.gold.fraud_boosted`

TRANSFORM (
  amount,
  LN(amount)                                              AS log_amount,
  category,

  EXTRACT(HOUR      FROM transaction_ts)                  AS hour,
  EXTRACT(DAYOFWEEK FROM transaction_ts)                  AS day_of_week,
  (EXTRACT(HOUR FROM transaction_ts) >= 22
   OR EXTRACT(HOUR FROM transaction_ts) <= 3)             AS is_night,

  DATE_DIFF(DATE(transaction_ts), customer_dob, YEAR)
    - IF(FORMAT_DATE('%m%d', DATE(transaction_ts))
          < FORMAT_DATE('%m%d', customer_dob), 1, 0)      AS age,

  -- Haversine em km. Escrito com ATAN2/SIN/COS/SQRT/POW de proposito:
  -- ST_DISTANCE e ST_GEOGPOINT NAO constam na lista de funcoes que o BigQuery
  -- aceita dentro do TRANSFORM ao exportar/implantar o modelo, e GEOGRAPHY e
  -- um tipo proibido nesse caminho. Com ST_* o modelo TREINA normalmente, mas
  -- o deploy no Vertex AI em outubro fica em risco. ACOS(-1) e o pi.
  2 * 6371.0 * ATAN2(
    SQRT(
      POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
      + COS(customer_lat * ACOS(-1) / 180)
        * COS(merchant_lat * ACOS(-1) / 180)
        * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2)
    ),
    SQRT(1 -
      ( POW(SIN((merchant_lat - customer_lat) * ACOS(-1) / 360), 2)
        + COS(customer_lat * ACOS(-1) / 180)
          * COS(merchant_lat * ACOS(-1) / 180)
          * POW(SIN((merchant_long - customer_long) * ACOS(-1) / 360), 2) )
    )
  )                                                       AS distance_km,

  city_pop,

  transaction_ts,
  is_fraud
)

OPTIONS (
  MODEL_TYPE               = 'BOOSTED_TREE_CLASSIFIER',
  BOOSTER_TYPE             = 'GBTREE',
  TREE_METHOD              = 'HIST',
  MAX_ITERATIONS           = 50,
  EARLY_STOP               = TRUE,
  SUBSAMPLE                = 0.85,
  MAX_TREE_DEPTH           = 8,

  INPUT_LABEL_COLS         = ['is_fraud'],
  AUTO_CLASS_WEIGHTS       = TRUE,

  DATA_SPLIT_METHOD        = 'SEQ',
  DATA_SPLIT_COL           = 'transaction_ts',
  DATA_SPLIT_EVAL_FRACTION = 0.20,

  MODEL_REGISTRY           = 'VERTEX_AI',
  VERTEX_AI_MODEL_ID       = 'fraudflow-boosted'
) AS
SELECT * EXCEPT (transaction_id, split_key)
FROM `{PROJECT_ID}.gold.ml_input`;
"""

In [ ]:
client.query(SQL_V2).result()
print("modelo v2 treinado")

## O teste do dia 8

In [ ]:
for nome in ['fraud_logistic', 'fraud_boosted']:
    m = client.get_model(f"{PROJECT_ID}.gold.{nome}")
    print(f"{nome:<16} {m.model_type}")

print("\nConfira em Vertex AI > Model Registry se aparecem:")
print("  fraudflow-logistic  e  fraudflow-boosted")

## Avaliação

Com fraude em 0,58% dos casos, **acuracia nao diz nada** — responder "nao e fraude"
para tudo ja acerta 99,4%. O slide lidera com recall, precision e F1.

In [ ]:
client.query(f"""
    SELECT 'v1 · logistica' AS modelo,
           ROUND(recall, 4) AS recall, ROUND(precision, 4) AS precision,
           ROUND(f1_score, 4) AS f1, ROUND(roc_auc, 4) AS roc_auc,
           ROUND(accuracy, 4) AS accuracy
    FROM ML.EVALUATE(MODEL `{PROJECT_ID}.gold.fraud_logistic`)
    UNION ALL
    SELECT 'v2 · boosted tree',
           ROUND(recall, 4), ROUND(precision, 4), ROUND(f1_score, 4),
           ROUND(roc_auc, 4), ROUND(accuracy, 4)
    FROM ML.EVALUATE(MODEL `{PROJECT_ID}.gold.fraud_boosted`)
    ORDER BY modelo
""").to_dataframe()

Matriz de confusao. Falso negativo e fraude que passou; falso positivo e cliente
legitimo bloqueado. Custam coisas diferentes, e e isso que o limiar decide.

In [ ]:
client.query(f"""
    SELECT * FROM ML.CONFUSION_MATRIX(MODEL `{PROJECT_ID}.gold.fraud_boosted`)
""").to_dataframe()

Importancia das features. Serve para conferir se as nove estao fazendo alguma
coisa, e quais sao decisivas.

In [ ]:
client.query(f"""
    SELECT feature,
           ROUND(importance_gain, 5)   AS ganho,
           ROUND(importance_weight, 5) AS peso,
           ROUND(importance_cover, 5)  AS cobertura
    FROM ML.FEATURE_IMPORTANCE(MODEL `{PROJECT_ID}.gold.fraud_boosted`)
    ORDER BY ganho DESC
""").to_dataframe()

Pesos da logistica. Permitem ler a direcao de cada efeito — coisa que a arvore
nao da, e que rende uma frase boa na apresentacao.

In [ ]:
client.query(f"""
    SELECT processed_input AS feature, ROUND(weight, 5) AS peso
    FROM ML.WEIGHTS(MODEL `{PROJECT_ID}.gold.fraud_logistic`)
    WHERE weight IS NOT NULL
    ORDER BY ABS(weight) DESC
    LIMIT 20
""").to_dataframe()

## A demo ao vivo

O modelo recebe **so os campos do contrato**, sem `is_fraud`. O rotulo entra
depois, por um JOIN, apenas para mostrar acerto e erro na tela.

Nao e firula: e a mesma separacao que a API de outubro vai ter.

In [ ]:
SQL_DEMO = f"""
-- ===========================================================================
-- 33 · Predição — a demo ao vivo
--
-- Repare na estrutura: o modelo recebe SÓ os campos do contrato, sem
-- is_fraud. O rótulo entra depois, por um JOIN em transaction_id, apenas para
-- mostrar acerto e erro na tela.
--
-- Isso não é firula. É a mesma separação que a API do Trabalho 2 vai ter:
-- lá o evento chega sem rótulo nenhum, e o modelo precisa decidir sozinho.
-- Escrever a demo assim prova que o contrato está sendo respeitado.
--
-- O recorte é o período de validação — os 20% finais por tempo, que o
-- DATA_SPLIT_METHOD='SEQ' reservou e o modelo nunca viu no treino.
-- ===========================================================================

WITH corte AS (
  SELECT APPROX_QUANTILES(transaction_ts, 5)[OFFSET(4)] AS inicio_validacao
  FROM `{PROJECT_ID}.gold.ml_input`
),

-- Só os campos do contrato. Nenhum rótulo passa por aqui.
evento AS (
  SELECT
    transaction_id,
    transaction_ts,
    customer_dob,
    customer_lat,
    customer_long,
    merchant_lat,
    merchant_long,
    amount,
    category,
    city_pop
  FROM `{PROJECT_ID}.gold.ml_input`, corte
  WHERE transaction_ts >= corte.inicio_validacao
),

pontuado AS (
  SELECT
    transaction_id,
    transaction_ts,
    amount,
    category,
    (SELECT p.prob
     FROM UNNEST(predicted_is_fraud_probs) AS p
     WHERE CAST(p.label AS STRING) = '1')                 AS fraud_score
  FROM ML.PREDICT(MODEL `{PROJECT_ID}.gold.fraud_boosted`,
                  (SELECT * FROM evento))
)

-- O rótulo só encosta na predição AGORA, depois da inferência.
SELECT
  FORMAT_TIMESTAMP('%d/%m %H:%M', p.transaction_ts)       AS quando,
  p.category                                              AS categoria,
  ROUND(p.amount, 2)                                      AS valor,
  ROUND(p.fraud_score, 4)                                 AS fraud_score,
  IF(p.fraud_score >= 0.30, 'FRAUD', 'LEGIT')             AS decisao,
  r.is_fraud                                              AS rotulo_real,
  IF((p.fraud_score >= 0.30) = (r.is_fraud = 1), 'ok', 'ERRO') AS resultado
FROM pontuado AS p
JOIN `{PROJECT_ID}.gold.ml_input` AS r
  USING (transaction_id)
ORDER BY p.fraud_score DESC
LIMIT 20;
"""

In [ ]:
client.query(SQL_DEMO).to_dataframe()

### A frase que salva a demo

Ao rodar esta consulta, diga em voz alta:

> *"este e o periodo de validacao, os 20% finais por tempo que o modelo nunca viu
> no treino, porque o split foi temporal"*

Responde antecipadamente a pergunta mais provavel da banca.

---

### Antes de fechar

1. Confira o **Billing** — o boosted tree e o unico custo imprevisivel
2. Baixe os quatro notebooks executados e comite, para o repositorio mostrar as
   saidas de verdade e nao celulas vazias
3. Nao rode o teardown antes do dia 11: o enunciado exige o pipeline em producao